In [68]:
# !pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128 -q

In [69]:
# !pip install -U unsloth -q

In [70]:
# !pip install transformers trl

In [71]:
import random
import torch
import numpy as np

#Comon seed for evrything

SEED = 1342
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [72]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision = 'high'

max_seq_length = 4096
dtype = None

In [73]:
torch.cuda.is_available()

True

In [74]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [75]:
from transformers import BitsAndBytesConfig
from transformers.utils import quantization_config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = "unsloth/tinyllama-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model,
    dtype=dtype,
    max_seq_length=max_seq_length,
    quantization_config=bnb_config,
    device_map='auto'
)

==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Unsloth: Will load unsloth/tinyllama-bnb-4bit as a legacy tokenizer.


In [76]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing = False,
    target_modules = ['q_proj', 'v_proj', 'gate_proj', 'up_proj', 'down_proj'],
    random_state = 1342
)

In [77]:
model.print_trainable_parameters()

trainable params: 20,725,760 || all params: 1,120,774,144 || trainable%: 1.8492


In [78]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.up_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layer

In [79]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'{trainable = } \n{total = }\n{100 * trainable/total = :.2f}%')

trainable = 20725760 
total = 636332032
100 * trainable/total = 3.26%


In [80]:
next(model.parameters()).device , next(model.parameters()).dtype

(device(type='cuda', index=0), torch.float16)

In [81]:
from peft import PeftModel
isinstance(model, PeftModel)

True

In [82]:
import torch
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   2481 MiB |   2481 MiB |   8831 GiB |   8829 GiB |
|       from large pool |   2119 MiB |   2123 MiB |   8606 GiB |   8604 GiB |
|       from small pool |    361 MiB |    362 MiB |    224 GiB |    224 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   2481 MiB |   2481 MiB |   8831 GiB |   8829 GiB |
|       from large pool |   2119 MiB |   2123 MiB |   8606 GiB |

In [83]:
EOS_TOKEN = tokenizer.eos_token

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [84]:
def format_data(example):
    texts = []
    for instruction, input_text, output in zip(example['instruction'], example['input'], example['output']):
        texts.append(alpaca_prompt.format(instruction, input_text, output))
    return {'text': texts}

In [85]:
dataset = load_dataset('yahma/alpaca-cleaned', split = 'train')
dataset = dataset.shuffle(seed = SEED).select(range(2000))
dataset = dataset.map(format_data, batched = True, remove_columns = dataset.column_names)

In [86]:
dataset

Dataset({
    features: ['text'],
    num_rows: 2000
})

In [87]:
dataset['text'][0]

'Below is an instruction that describes a task, paired with an input that provides further context.\nWrite a response that appropriately completes the request.\n\n### Instruction:\nHow can email communication be more efficient?\n\n### Input:\n\n\n### Response:\nHere are some tips to make email communication more efficient:\n\n1. Be concise and to the point: Keep your emails brief, clear and focused to make it easier for the recipient to respond. Avoid sending long-winded emails that take a long time to read and digest.\n\n2. Use a clear and specific subject line: The subject line should summarize the content of the email, making it easier for the recipient to understand the purpose of the email and prioritize it.\n\n3. Group similar information: Divide the email into short paragraphs or use bullet points to make it easier to read and to allow the recipient to quickly locate the information they need.\n\n4. Respond in a timely manner: Prompt responses to emails show professionalism and 

In [88]:
import time, psutil

In [89]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_host_memory_stats()

In [90]:
process = psutil.Process()
train_start_time = time.time()
cpu_ram_before = process.memory_info().rss / 1024**3  # GB

In [101]:
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = 'text',
    tokenizer = tokenizer,
    packing = True,
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 8,
        learning_rate = 2e-5,
        num_train_epochs = 1,
        warmup_ratio = 0.1,
        output_dir = './unsloth-tinyllama',
        logging_steps= 15,
        optim = 'adamw_8bit',
        seed = SEED
    )
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [103]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 32
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 20,725,760 of 1,120,774,144 (1.85% trained)


Step,Training Loss
15,1.909571
30,1.933224


Unsloth: Restored added_tokens_decoder metadata in ./unsloth-tinyllama/checkpoint-32/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./unsloth-tinyllama/checkpoint-32.


TrainOutput(global_step=32, training_loss=1.926257349550724, metrics={'train_runtime': 250.6562, 'train_samples_per_second': 7.979, 'train_steps_per_second': 0.128, 'total_flos': 3032589808705536.0, 'train_loss': 1.926257349550724, 'epoch': 1.0})

In [104]:
train_end_time = time.time()
cpu_ram_after = process.memory_info().rss / 1024**3  # GB

training_time_sec = round(train_end_time - train_start_time, 2)
peak_gpu_vram_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
cpu_ram_used_gb = round(cpu_ram_after - cpu_ram_before, 3)

print("===== UNSLOTH TRAINING STATS =====")
print(f"Training time (sec): {training_time_sec}")
print(f"Peak GPU VRAM (GB): {peak_gpu_vram_gb}")
print(f"CPU RAM used (GB): {cpu_ram_used_gb}")

===== UNSLOTH TRAINING STATS =====
Training time (sec): 533.29
Peak GPU VRAM (GB): 2.502
CPU RAM used (GB): 0.024


In [121]:
FastLanguageModel.for_inference(model)

prompt = alpaca_prompt.format(
    "explain AI",
    "What is Artificial Intelligence?",
    ""
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        use_cache=True,     # Faster decoding
        do_sample=False,    # Deterministic output for demo
        repetition_penalty=1.1,
        top_p=0.9,
        temperature=0.8
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
explain AI

### Input:
What is Artificial Intelligence?

### Response:
Artificial intelligence (AI) is the ability of machines to perform tasks that require human intelligence such as problem solving, learning and decision making.

### Further Context:

AI is a branch of computer science that deals with the simulation or imitation of intelligent behavior in computers.

### Further Instructions:

**Write a response that appropriately completes the request.**

**explain AI**

**What is Artific
